# Example 1: CRAM injection in three steps

This example shows the CRAM injection functions of SK-RAFT in three steps on one design. Each step is more selective than the step before it:

1. **Full-device CRAM.** The script reads back all the CRAM frames. Then it upsets random bits in these frames.
2. **Essential bits.** The server selects each bit from the design's essential bits. The server reads the essential bits from the EBD file.
3. **The essential bits of one pblock.** This step is the same as step 2, but in one pblock at a time: first `pblock_slow_heartbeat`, then `pblock_fast_heartbeat`.

The script keeps each upset for 100 ms, then flips the same bit again to correct it. Each step, and each pblock in step 3, continues until you press Enter. The script reprograms the FPGA before steps 2 and 3. Monitor the LEDs on the board to see the effect of each upset.

## Design

<img src="images/example1_targeted_top_abstract.svg" alt="Block diagram of targeted_top. The safe clock generation feeds a slow heartbeat and a fast heartbeat. Each heartbeat is in its own pblock and drives an LED." />

The design for this example is [`targeted_top`](../hw/sources_1/new/targeted_top.vhd). It has a fast heartbeat counter and a slow heartbeat counter, each in its own pblock. The XDC file of the design declares the two pblocks, for example:

```tcl
create_pblock pblock_fast_heartbeat
add_cells_to_pblock [get_pblocks pblock_fast_heartbeat] [get_cells fast_hb_inst]
resize_pblock [get_pblocks pblock_fast_heartbeat] -add {SLICE_X54Y91:SLICE_X57Y99}
```

A pblock keeps the cells of its instance in the given area. Two more properties control the pblock:

- `set_property EXCLUDE_PLACEMENT true [get_pblocks ...]` prevents the placement of other cells in the pblock.
- `set_property CONTAIN_ROUTING true [get_pblocks ...]` keeps the routing of the pblock inside the pblock.

The examples do not use `CONTAIN_ROUTING`, to keep them simple.

## The calls of the example

| Call | Use in the example |
| --- | --- |
| `SkraftFpga.connect(ip=..., port=...)`, `client.attach(SERIAL_NUMBER)` | Connect to the SK-RAFT server and attach to the device, in `with` statements that release the device and close the connection. |
| `Design(bit, ebd, xdc)`, `dev.uploadDesignToServer(design)`, `dev.programFPGA(design)` | Give the design files, upload them to the server and program the FPGA. |
| `dev.configure_seed(0)` | Set the seed of the random injections. The same seed gives the same upsets in each run. |
| `dev.readback(ReadbackOption.READBACK)` | Step 1: read back all the CRAM frames. The script selects a random bit of these frames. |
| `dev.inject_at(far, word, bit, TargetMemory.CRAM)` | Step 1: flip the selected bit. In all steps: flip the bit again after 100 ms, to correct the upset. |
| `dev.bit_count(TargetMemory.ESSENTIAL_CRAM)`, `dev.bit_count(TargetMemory.ESSENTIAL_CRAM, pblock=pblock)` | Steps 2 and 3: get the number of essential bits of the device, and of each pblock. |
| `dev.inject_random(TargetMemory.ESSENTIAL_CRAM)` | Step 2: inject one upset into a random essential bit. |
| `dev.inject_random(TargetMemory.ESSENTIAL_CRAM, pblock=pblock)` | Step 3: inject one upset into a random essential bit of one pblock. |

## Run the example


### 0: Install the module

This will restart the kernel, a popup will show up, just ignore it.

In [ ]:
%pip install ../../api/python

import IPython; IPython.get_ipython().kernel.do_shutdown(restart=True)

### 1: Extract the design and bitstream files

On Linux

In [ ]:
%%bash
cd ../hw/design_examples/
./unzip_designs.sh

On Windows

In [ ]:
%%script pwsh
cd ..\hw\design_examples\
.\unzip_designs.bat

### 2: Add the imports

In [ ]:
import random
import time

from skraft import Design, TargetMemory, ReadbackOption, SkraftError, SkraftFpga

SK_RAFT_SERVER_IP = "127.0.0.1"
SK_RAFT_SERVER_PORT = 50051

### 3: Start the server

In [ ]:
%%script bash --bg
sk-raft

### 4: List devices
Connect the devices you want to test before running list devices.

In [ ]:
def main() -> int:
    try:
        # SK-RAFT: open the connection. Then list the connected boards.
        with SkraftFpga.connect(ip=SK_RAFT_SERVER_IP,
                                port=SK_RAFT_SERVER_PORT) as client:
            devices = client.get_devices()
    except SkraftError as e:
        print(f"Cannot list the boards: {e}")
        return 1
    except KeyboardInterrupt:
        print("Interrupted.")
        return 130

    if not devices:
        print("No board is connected to the SK-RAFT server.")
        return 1
    print(f"{'SERIAL NUMBER':16s} PART NUMBER")
    for device in devices:
        print(f"{device.serial_number:16s} {device.part_number}")
    return 0


if __name__ == "__main__":
    main()

### 5: Setup the code
Don't forget to change the Serial Number (`SERIAL_NUMBER`) and the Board (`BOARD`) to the ones you desire from the list!

In [ ]:
SERIAL_NUMBER = "210183BA2524A"

# The design for all steps: two heartbeat counters (LD0, LD1).
# The script gives all three files at the start:
#   The bitstream (.bit): to program the device.
#   The EBD file (.ebd): to find which bits are essential.
#   The XDC file (.xdc): to find the pblocks.
# uploadDesignToServer() sends the files to the server. The client sends
# only the file name of each path, not the directory.
#
# BOARD is the board of your target device: "basys3" or "zedboard", or
# the name of your own board (see hw/README.md). The script reads the
# design files from hw/design_examples/<BOARD>/targeted_top/. Extract
# design_examples.zip there, or build the design with create_project.tcl.
# The paths are relative to src/. Run the script from that folder.
BOARD = "basys3"
DESIGN_DIR = f"../hw/design_examples/{BOARD}/targeted_top"

design = Design(
    bit=f"{DESIGN_DIR}/targeted_top.bit",
    ebd=f"{DESIGN_DIR}/targeted_top.ebd",
    xdc=f"{DESIGN_DIR}/board.xdc",
)

# A fixed seed gives the same sequence of step 1 injections in each run.
# In steps 2 and 3, the server selects the bits.
rng = random.Random(42)

# A background thread monitors the Enter key. Thus, the injection loop
# below does not wait for input. Press Enter at any time to stop the
# current step early.
client = SkraftFpga.connect(ip=SK_RAFT_SERVER_IP, port=SK_RAFT_SERVER_PORT)
dev = client.attach(SERIAL_NUMBER)
# Setup the SEED for deterministic behaviour. 
dev.configure_seed(0)

### 6: Program the board...
...and upload the design to the server because it needs it for some operations.

In [ ]:
dev.uploadDesignToServer(design)
dev.programFPGA(design)
print(f"programmed {SERIAL_NUMBER} with {design.bit}")

### 7: Run the experiments!
The `N_OF_INJECTIONS` is the number of fault injections the experiment will do.

In [ ]:
# ----------------------------------------------------------- #
# Step 1: random bits, anywhere in the configuration RAM.
# ----------------------------------------------------------- #

N_OF_INJECTIONS = 1000

print("\nStep 1: random bits, anywhere. ")

# This is a full random CRAM injection routine, with all
# steps visible in this script. The script reads back all
# frames of the connected device. Then the script selects
# the frame, word and bit to flip.
frames = dev.readback(ReadbackOption.READBACK)
print(f"frames read back: {len(frames)}")
time.sleep(1)
# The number of injections in this step.
n = N_OF_INJECTIONS
while n!=0:
    # Inject a random CRAM upset.
    frame = frames[rng.randint(0, len(frames) - 1)]
    # The number of words in a frame depends on the
    # architecture.
    word_index = rng.randint(0, len(frame.words) - 1)
    # A word has 32 bits, so the bit index is 0 to 31.
    bit_index = rng.randint(0, 31)
    report = dev.inject_at(frame.far, word_index, bit_index, TargetMemory.CRAM)
    print(f"{n + 1:3d}: far {report.far:#010x} word {report.word_index:3d} bit {report.bit_index:2d}")
    n -= 1
    # Keep the upset for 100 ms.
    time.sleep(0.1)
    # Flip the same bit again to correct the upset.
    dev.inject_at(frame.far, word_index, bit_index, TargetMemory.CRAM)

In [ ]:
# ----------------------------------------------------------- #
# Step 2: essential bits only.
#
# The EBD file lists the bits that the design depends on.
# Limit the injection to these bits. Then each flip hits
# one of these bits.
# ----------------------------------------------------------- #

N_OF_INJECTIONS = 1000

print(f"\nStep 2: essential bits only ({dev.bit_count(TargetMemory.ESSENTIAL_CRAM)})")
# Program the FPGA again to remove the upsets of step 1.
dev.programFPGA(design)
n = N_OF_INJECTIONS
while n!=0:
    # This time, the server selects the location from the
    # essential bits in the EBD file.
    report = dev.inject_random(TargetMemory.ESSENTIAL_CRAM)
    print(f"{n + 1:3d}: far {report.far:#010x} word {report.word_index:3d} bit {report.bit_index:2d}")
    n -= 1
    # Keep the upset for 100 ms.
    time.sleep(0.1)
    dev.inject_at(report.far, report.word_index, report.bit_index, TargetMemory.CRAM)

In [ ]:
# ----------------------------------------------------------- #
# Step 3: one pblock at a time.
#
# A pblock is a region of the device that contains only one
# part of the design. In this design, each pblock contains
# one of the two heartbeat counters. When you limit the
# injection to a pblock, you can test one part of the design
# alone.
# ----------------------------------------------------------- #

N_OF_INJECTIONS = 1000

print("\nStep 3: one pblock at a time.")
# Program the FPGA again to remove the upsets of step 2.
dev.programFPGA(design)
# The names of the two pblocks, as declared in the XDC file.
pblocks = ["pblock_slow_heartbeat", "pblock_fast_heartbeat"]
for pblock in pblocks:
    n_essential = dev.bit_count(TargetMemory.ESSENTIAL_CRAM, pblock=pblock)
    print(f"\n{pblock}: essential bits ({n_essential}). "f"Press Enter to move on.")
    time.sleep(1)
    n = N_OF_INJECTIONS
    while n!=0:
        # This time, the server selects the location from
        # the essential bits in the EBD file and from the
        # pblock range in the XDC file.
        report = dev.inject_random(TargetMemory.ESSENTIAL_CRAM, pblock=pblock)
        print(f"{n + 1:3d}: far {report.far:#010x} "f"word {report.word_index:3d} "f"bit {report.bit_index:2d}")
        n -= 1

        # Keep the upset for 100 ms.
        time.sleep(0.1)
        dev.inject_at(report.far, report.word_index, report.bit_index, TargetMemory.CRAM)